In [14]:
import pandas as pd  
import os  
import io
from datetime import date, datetime  
from dotenv import load_dotenv  
import msoffcrypto
import calendar
import warnings
from dateutil.relativedelta import relativedelta
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError
from openpyxl.utils import get_column_letter


warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [5]:
# ---------- Settings ----------  
FILES = {  
    2025: 'L:/2025 Pilar Plant Files/Monthly Expense Reports/12. December/MEETH Financial Statements 12-2025.xlsx',
    2026: 'L:/2026 Pilar Plant Files/Monthly Expense Reports/9. September/MEETH/MEETH Financial Statements 09-2026 v2.xlsx',  
}  
SHEET = 'OP Summary'  
DEPT_KEYWORD = 'Chemo'          # matches 'Private Amb - Chemotherapy'

SECTIONS = ['Volume', 'Revenue']  
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',  
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']      # no YTD  
METRICS = ['Actual', 'Budget', 'Variance']

# 'january' -> 'Jan', 'jan' -> 'Jan', 'sept' -> 'Sep'  
_MONTH_MAP = {}  
for i, abbr in enumerate(MONTHS, start=1):  
    _MONTH_MAP[abbr.lower()] = abbr  
    _MONTH_MAP[calendar.month_name[i].lower()] = abbr  
_MONTH_MAP['sept'] = 'Sep'


# ---------- Helpers ----------  
def strip_cell(x):  
    return x.strip() if isinstance(x, str) else x


def month_label(x):  
    """Normalize a row label to 'Jan'..'Dec', or None if it isn't a month."""  
    if isinstance(x, (pd.Timestamp, datetime)):  
        return x.strftime('%b')  
    if not isinstance(x, str):  
        return None  
    return _MONTH_MAP.get(x.strip().lower())


def to_num(col):  
    s = (col.astype(str)  
            .str.strip()  
            .str.replace(r'[,$%]', '', regex=True)  
            .str.replace(r'^\((.*)\)$', r'-\1', regex=True)  
            .replace({'-': '0', '–': '0', 'nan': None, 'None': None, '': None}))  
    return pd.to_numeric(s, errors='coerce')


def find_section_rows(raw, sections):  
    """Find the row index of each section label (case-insensitive, any column)."""  
    lowered = raw.apply(lambda c: c.map(lambda v: v.lower() if isinstance(v, str) else v))  
    starts = []  
    for s in sections:  
        # Prefer an exact match in column A, then an exact match anywhere,  
        # then a "starts with" match anywhere.  
        col0 = lowered.iloc[:, 0]  
        hits = col0.index[col0 == s.lower()]  
        if hits.empty:  
            hits = lowered.index[lowered.eq(s.lower()).any(axis=1)]  
        if hits.empty:  
            mask = lowered.apply(lambda c: c.astype(str).str.startswith(s.lower()))  
            hits = lowered.index[mask.any(axis=1)]  
        if hits.empty:  
            raise ValueError(f"Section '{s}' not found on sheet '{SHEET}'")  
        starts.append(hits[0])  
    return starts

def parse_block(block, section_name=''):  
    """Return a long DataFrame: month, entity, department, metric, value, excel_col, group."""  
    block = block.reset_index(drop=True)

    hdr_hits = block.index[block.eq('Actual').any(axis=1)]  
    if hdr_hits.empty:  
        raise ValueError(f"No 'Actual' header row found in section '{section_name}'")  
    hdr = hdr_hits[0]  
    if hdr < 2:  
        raise ValueError(f"Header row in '{section_name}' has no room for entity/dept rows above it")

    metric_row = block.iloc[hdr]  
    group = (metric_row == 'Actual').cumsum()          # one group per Actual/Budget/Variance set  
    is_metric = metric_row.isin(METRICS) & (group > 0)  
    metric_cols = is_metric[is_metric].index  
    first_col = metric_cols[0]

    # Department: first non-blank label *within* each group (no bleeding into neighbours)  
    dept = block.iloc[hdr - 1].groupby(group).transform('first')

    # Entity: merged across several groups -> forward-fill, then first per group  
    entity_row = block.iloc[hdr - 2].copy()  
    entity_row.loc[first_col:] = entity_row.loc[first_col:].ffill()  
    entity = entity_row.groupby(group).transform('first')

    # Month rows: first Jan..Dec run below the header  
    labels = block.iloc[:, 0].map(month_label)  
    month_rows = labels[(labels.index > hdr) & labels.notna()]  
    month_rows = month_rows[~month_rows.duplicated(keep='first')]  
    if month_rows.empty:  
        sample = block.iloc[hdr + 1:hdr + 6, 0].tolist()  
        raise ValueError(f"No month rows in '{section_name}'. First labels under header: {sample}")

    vals = block.loc[month_rows.index, metric_cols].apply(to_num)  
    vals.index = pd.Index(month_rows.values, name='month')

    col_info = pd.DataFrame({  
        'col':        metric_cols,  
        'entity':     entity[metric_cols].values,  
        'department': dept[metric_cols].values,  
        'metric':     metric_row[metric_cols].values,  
    })

    long = (vals.reset_index()  
                .melt(id_vars='month', var_name='col', value_name='value')  
                .merge(col_info, on='col')  
                .drop(columns='col'))  
    return long


def load_chemo(path, year):  
    raw = pd.read_excel(path, sheet_name=SHEET, header=None)  
    raw = raw.apply(lambda col: col.map(strip_cell))

    starts = find_section_rows(raw, SECTIONS)  
    order = sorted(range(len(starts)), key=lambda i: starts[i])  
    ends = {}  
    for k, i in enumerate(order):  
        ends[i] = starts[order[k + 1]] if k + 1 < len(order) else len(raw)

    parts = []  
    for i, name in enumerate(SECTIONS):  
        df = parse_block(raw.iloc[starts[i]:ends[i]], f'{year} {name}')  
        df.insert(0, 'section', name)  
        parts.append(df)  
    out = pd.concat(parts, ignore_index=True)

    # Keep Chemo only  
    mask = out['department'].astype(str).str.contains(DEPT_KEYWORD, case=False, na=False)  
    if not mask.any():  
        depts = sorted(out['department'].dropna().astype(str).unique())  
        raise ValueError(f"{year}: no department containing '{DEPT_KEYWORD}'. Found: {depts}")  
    out = out[mask].dropna(subset=['value'])

    # Report label collisions (the thing that broke unstack)  
    key = ['section', 'entity', 'department', 'metric', 'month']  
    dups = out[out.duplicated(key, keep=False)]  
    if not dups.empty:  
        print(f"\n{year}: Chemo labels appear in more than one column group:")  
        print(dups.drop_duplicates(['section'])  
                  [['section', 'entity', 'department', 'metric']]  
                  .to_string(index=False))

    out.insert(0, 'year', year)  
    print(f'{year}: loaded {len(out)} rows')  
    return out


# ---------- Load all files and combine ----------  
chemo_long = pd.concat(  
    [load_chemo(path, year) for year, path in FILES.items()],  
    ignore_index=True,  
)

chemo_long['month'] = pd.to_datetime(  
    chemo_long['year'].astype(str) + '-' + chemo_long['month'] + '-01',  
    format='%Y-%b-%d',  
)

chemo_long = (chemo_long  
              .sort_values(['year', 'section', 'month'])  
              .reset_index(drop=True))

2025: loaded 72 rows
2026: loaded 54 rows


In [7]:
# ---------------- function to read protected excel ----------------

def read_protected_excel(file_path, password, sheet_name, skiprows):  
    """Read a single sheet from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    workbook.seek(0)  
    df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
    return df

# ---------------- config ----------------

load_dotenv()  
password = os.getenv("EPIC_INFUSION_PW")

epic_path = 'L:/2026 Pilar Plant Files/Requests from Departments/EPIC/Transactions (HB)_2026-11-12T15_26_00.xlsx'
sheet_name = 'Data'
skiprows = 10

try:  
    epic_infusion = read_protected_excel(epic_path, password, sheet_name, skiprows)  
except Exception as e:  
    print(f"  ERROR: {e}")

print(f"\n--- Summary ---")  
print(f"Epic Data: {epic_infusion.shape}")  


    (encrypted)

--- Summary ---
Epic Data: (13884, 17)


In [9]:
infusion_cases = (
    epic_infusion[['Patient Epic ID', 'Service Date', 'Charge Code', 'Posted Amount ($)', 'Hospital Account ID']]
    .drop_duplicates()
    .rename(columns={'Charge Code': 'CPT Code', 'Posted Amount ($)': 'Amount'})
)

display(infusion_cases)

,Patient Epic ID,Service Date,CPT Code,Amount,Hospital Account ID
0,Z10047959,2026-01-21,96365,3200,1055000023836
1,Z10047959,2026-01-12,96365,3200,1055000023835
2,Z10047959,2026-01-29,96365,3200,1055000023837
3,Z10074330,2026-01-15,96365,3200,1055000020536
4,Z10113116,2026-01-09,96365,3200,1055000022105
...,...,...,...,...,...
13879,Z7907020,2026-10-20,99195,1283,1055000073636
13880,Z7921814,2026-10-02,99195,1283,1055000070425
13881,Z8097275,2026-10-27,99195,1283,1055000063413
13882,Z8179071,2026-10-06,99195,1283,1055000060967


In [21]:
infusion_types = pd.DataFrame(  
    [ (96401, 'Injection/Transfusion'),
      (96402, 'Injection/Transfusion'),
      (96409, 'Infusion Chemotherapy'),
      (96411, 'Infusion Chemotherapy'),
      (96413, 'Infusion Chemotherapy'),
      (36415, 'Labs'),
      (36430, 'Injection/Transfusion'),
      (96360, 'Infusion Non-Chemotherapy'),
      (96365, 'Infusion Non-Chemotherapy'),
      (96372, 'Injection/Transfusion'),
      (96374, 'Injection/Transfusion'),
      (96377, 'Injection/Transfusion'),
      (96523, 'Injection/Transfusion'),
      (99195, 'Phlebotomy')
           ],  
    columns=['CPT Code', 'Category']
)

#96413 > 96409 > 96401 > 96402 > 96365 > 96374 > 96360 > 96372

# ---------- Hierarchy (highest priority first) ----------  
HIERARCHY = [96413, 96411, 96409, 96401, 96402, 96365, 96360, 96374, 96372, 96377, 96523, 36430, 36415, 99195]  
RANK = {code: i for i, code in enumerate(HIERARCHY, start=1)}

PATIENT_COL = 'Hospital Account ID'     # change to your actual column name  
GROUP_COLS = [PATIENT_COL]          # add a date/encounter column for per-visit (see below)

# Make sure CPT codes are numeric on both sides, or the merge silently fails  
infusion_cases['CPT Code'] = pd.to_numeric(infusion_cases['CPT Code'], errors='coerce').astype('Int64')  
infusion_types['CPT Code'] = infusion_types['CPT Code'].astype('Int64')

inf_cat = pd.merge(infusion_cases, infusion_types, how='left', on='CPT Code')

# ---------- Rank and flag ----------  
inf_cat['Hierarchy Rank'] = inf_cat['CPT Code'].map(RANK)   # codes not in the list -> NaN

primary_idx = (inf_cat.dropna(subset=['Hierarchy Rank'])  
                      .groupby(GROUP_COLS)['Hierarchy Rank']  
                      .idxmin())                             # one row per group, even with repeats

inf_cat['Primary Flag'] = 'Non-Primary'  
inf_cat.loc[primary_idx, 'Primary Flag'] = 'Primary'

inf_cat['Month'] = inf_cat['Service Date'].dt.to_period('M').dt.to_timestamp()

inf_grouped = inf_cat[inf_cat['Primary Flag'] == 'Primary'].groupby(['Category', 'Month'])['Hospital Account ID'].nunique().reset_index()

display(inf_grouped)

,Category,Month,Hospital Account ID
0,Infusion Chemotherapy,2026-01-01,290
1,Infusion Chemotherapy,2026-02-01,278
2,Infusion Chemotherapy,2026-03-01,306
3,Infusion Chemotherapy,2026-04-01,329
4,Infusion Chemotherapy,2026-05-01,314
5,Infusion Chemotherapy,2026-06-01,329
6,Infusion Chemotherapy,2026-07-01,319
7,Infusion Chemotherapy,2026-08-01,300
8,Infusion Chemotherapy,2026-09-01,299
9,Infusion Chemotherapy,2026-10-01,311


In [22]:

# write dataset to file
today = date.today().strftime('%Y%m%d')
current_time = datetime.now().strftime('%b %d, %Y %I:%M %p')  
file_path = f'C:/Users/kbixby/OneDrive - Northwell Health/Documents/Financials/MEETH Infusion/infusion_data_{today}.xlsx'

with pd.ExcelWriter(file_path, engine='openpyxl') as writer:  
    chemo_long.to_excel(writer, sheet_name='financials', index=False)
    inf_grouped.to_excel(writer, sheet_name='volume', index=False)

print(f"{file_path} saved at {current_time}")  


C:/Users/kbixby/OneDrive - Northwell Health/Documents/Financials/MEETH Infusion/infusion_data_20261119.xlsx saved at Nov 19, 2026 10:15 AM
